# 🏙️ São Paulo Housing Analytics — Desafio SYNAPTA 2026
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gui-santos-dev/synapta/blob/main/Sao_Paulo_Housing_Analytics.ipynb)

> **Análise Exploratória de Dados, Auditoria de Qualidade e Modelagem Geoespacial sobre Preços de Aluguel na Cidade de São Paulo**  
> **Processo Seletivo SYNAPTA — Laboratório de Dados e Inteligência Artificial | Edital nº 1/2026 — UNINASSAU Maceió**

---

### 👤 Identificação do Candidato e do Projeto
| Item | Descrição |
| :--- | :--- |
| **Candidato** | **Guilherme Domingos dos Santos** *(Candidato nº 11 — Edital nº 1/2026)* |
| **Instituição** | Centro Universitário Maurício de Nassau — **UNINASSAU Maceió** |
| **Curso** | Bacharelado em Ciência da Computação |
| **Opção Escolhida** | **Opção 1 — Análise de preços de aluguel em São Paulo (*Sao Paulo Housing Prices*)** |
| **Dataset** | 11.657 anúncios imobiliários coletados na cidade de São Paulo |
| **Pergunta Orientadora** | *"Quais características parecem acompanhar os maiores e os menores preços de aluguel e o que os dados não permitem concluir?"* |

---
### 📋 Roteiro da Análise no Notebook
1. **Configuração e Carregamento dos Dados:** Importação de bibliotecas analíticas e download automático da base.
2. **Auditoria de Qualidade e Limpeza (Data Cleaning):** Inspeção de nulos, inconsistências (área zerada) e duplicatas parciais.
3. **Engenharia de Features e Agrupamento Territorial:** Mapeamento dos bairros nas 5 Macrozonas de São Paulo e cálculo de métricas ($R\$/m^2$).
4. **Estatística Descritiva Robusta:** Comparativo Média vs. Mediana e estudo da assimetria positiva dos preços.
5. **Visualizações Gráficas Avançadas (Plotly & Seaborn):** Distribuição por tipo, quartos, vagas e correlação área x preço.
6. **Mapeamento Geoespacial Interativo com Folium:** Visualização espacial dos bairros e aluguel médio em São Paulo.
7. **Síntese Técnica e Resposta à Pergunta Orientadora:** Conclusões e limitações metodológicas dos dados.
8. **(Opcional) Execução do Dashboard Web Flask:** Instruções para abrir o servidor web local diretamente pelo Colab.


### 1. Configuração do Ambiente e Importação de Bibliotecas

In [ ]:
# Instalação de bibliotecas complementares caso necessário no ambiente Colab
!pip install -q pandas numpy matplotlib seaborn plotly folium requests

import os
import sys
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import folium
from folium.plugins import FastMarkerCluster

# Configurações visuais elegantes para gráficos estáticos
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print("✅ Ambiente configurado com sucesso! Bibliotecas carregadas.")


### 2. Carregamento dos Dados
O script abaixo verifica automaticamente se o arquivo `data.csv` já existe localmente (na pasta `dados_extraidos/` ou raiz do repositório clonado). Se executado diretamente no Google Colab sem clone prévio, ele efetua o download automático da fonte pública ou repositório.


In [ ]:
### 3. Auditoria de Qualidade e Tratativas de Prepara??o dos Dados (Se??o 3 do Relat?rio T?cnico)

Conforme documentado no Relat?rio T?cnico do projeto, a prepara??o dos dados combinou a manipula??o manual em planilha eletr?nica e a valida??o program?tica em Python:
- **1. Valores Ausentes (`0` registros):** *"N?o h? valores ausentes na planilha. At? a ?ltima linha com dados est? preenchida de acordo."* (11.657 registros completos nas 8 colunas).
- **2. Identifica??o e Exclus?o de Duplicatas (`2 pares`):** *"Existem alguns pares, 2 pares para ser exato, contendo o mesmo endere?o, mas valores diferentes. Por possu?rem as mesmas caracter?sticas eu decidi apagar um dos registros de cada par para manter os dados mais limpos. Somente igualar o valor e considerar quartos diferentes do mesmo lugar ia acabar gerando duplicatas exatas."*
- **3. Inconsist?ncias e Valores At?picos (Metragens Zeradas ou Incompat?veis):** *"Apaguei as linhas com metragens zeradas ou incompat?veis com o porte do im?vel. Como n?o consigo ter a no??o da metragem exata, ? interessante apagar do que presumir uma informa??o de um dado inconsistente."* (Incluindo 2 registros com `area = 0 m?` na Bela Vista `R$ 2.800` e no Br?s `R$ 1.700`, filtrados no c?lculo de R$/m? sem presumir ?rea artificial).
- **4. Consist?ncia Financeira (`total < rent`):** `0` registros inconsistentes (100% com `total >= rent`).
- **5. Pulveriza??o de Localidades (`340` bairros com `n = 1`):** Aplicado corte m?nimo de representatividade (`n >= 15` an?ncios) nos rankings.
- **6. Regi?o Metropolitana (`232` an?ncios):** Segregados explicitamente das 5 Macrozonas do Munic?pio de S?o Paulo.


# C?pia para auditoria e valida??o estat?stica
df = df_raw.copy()
if 'rent_price' not in df.columns and 'rent' in df.columns:
    df['rent_price'] = df['rent']

print('--- ?? 1. Verifica??o de Valores Ausentes (NULL / Vazios) ---')
print(df.isnull().sum())

print('\n--- ?? 2. Inspe??o de Metragens Zeradas (area <= 0 m?) ---')
anomalias_area = df[df['area'] <= 0]
print(f'Registros com ?rea <= 0 m? (Bela Vista e Br?s): {len(anomalias_area)}')
display(anomalias_area)

print('\n--- ?? 3. Verifica??o de Consist?ncia Financeira (total < rent) ---')
print(f"Registros com total < rent: {(df['total'] < df['rent_price']).sum()}")

# C?lculo de R$/m? filtrando metragens zeradas (sem presumir ?rea artificial)
df_clean = df.copy()
df_clean['preco_m2'] = np.where(df_clean['area'] > 0, df_clean['rent_price'] / df_clean['area'], np.nan)

print(f"\n? Base auditada: {len(df_clean):,} registros | Aluguel M?dio: R$ {df_clean['rent_price'].mean():,.2f} | Mediana: R$ {df_clean['rent_price'].median():,.2f}")


In [ ]:
# Cópia para higienização
df = df_raw.copy()
if 'rent_price' not in df.columns and 'rent' in df.columns:
    df['rent_price'] = df['rent']

print("--- 🔍 1. Verificação de Dados Faltantes (Valores Nulos) ---")
print(df.isnull().sum())

print("\n--- 🔍 2. Inspeção de Anomalias Físicas (Área <= 0) ---")
anomalias_area = df[df['area'] <= 0]
print(f"Registros com área <= 0 m²: {len(anomalias_area)}")
display(anomalias_area)

print("\n--- 🔍 3. Verificação de Duplicatas Exatas e Parciais ---")
dup_exatas = df.duplicated().sum()
dup_parciais = df.duplicated(subset=['address', 'district', 'area', 'bedrooms', 'garage', 'type', 'rent_price']).sum()
print(f"Duplicatas exatas: {dup_exatas}")
print(f"Duplicatas parciais (mesmo imóvel, área, cômodos e preço): {dup_parciais}")

# Tratamento: Remoção das anomalias físicas severas (área zero)
df_clean = df[df['area'] > 0].copy()

# Cálculo do preço por metro quadrado (R$/m²)
df_clean['preco_m2'] = df_clean['rent_price'] / df_clean['area']

print(f"\n✅ Base limpa final: {len(df_clean):,} registros válidos mantidos.")


### 4. Engenharia de Features e Agrupamento Territorial (Macrozonas de SP)

A cidade de São Paulo possui forte diferenciação socioeconômica entre suas regiões. Mapeamos os distritos e bairros nas **5 Macrozonas Oficiais** da capital paulista:
- **Centro** (Sé, República, Bela Vista, Consolação, Santa Cecília, Higienópolis...)
- **Zona Oeste** (Pinheiros, Itaim Bibi, Jardins, Perdizes, Vila Madalena, Butantã, Morumbi...)
- **Zona Sul** (Vila Mariana, Saúde, Moema, Santo Amaro, Campo Limpo, Jabaquara...)
- **Zona Norte** (Santana, Tucuruvi, Casa Verde, Freguesia do Ó, Tremembé...)
- **Zona Leste** (Mooca, Tatuapé, Belém, Itaquera, Penha, São Mateus...)
- **Região Metropolitana / Outras** (Municípios vizinhos presentes na base, como Barueri, Guarulhos, Osasco...)


In [ ]:
# Dicionário de classificação das principais localidades nas Macrozonas de São Paulo
MAPA_REGIOES = {
    # Centro
    "Centro": "Centro", "Centro Histórico de São Paulo": "Centro", "Sé": "Centro", 
    "República": "Centro", "Bela Vista": "Centro", "Consolação": "Centro", 
    "Santa Cecília": "Centro", "Santa Cecilia": "Centro", "Vila Buarque": "Centro", 
    "Higienópolis": "Centro", "Campos Elíseos": "Centro", "Bom Retiro": "Centro", 
    "Liberdade": "Centro", "Cambuci": "Centro", "Aclimação": "Centro", "Brás": "Centro",
    
    # Zona Oeste
    "Pinheiros": "Zona Oeste", "Itaim Bibi": "Zona Oeste", "Jardim Paulista": "Zona Oeste",
    "Cerqueira César": "Zona Oeste", "Jardim Europa": "Zona Oeste", "Jardim América": "Zona Oeste",
    "Jardim Paulistano": "Zona Oeste", "Vila Olímpia": "Zona Oeste", "Vila Nova Conceição": "Zona Oeste",
    "Vila Madalena": "Zona Oeste", "Alto de Pinheiros": "Zona Oeste", "Perdizes": "Zona Oeste",
    "Vila Pompéia": "Zona Oeste", "Pompeia": "Zona Oeste", "Sumaré": "Zona Oeste", 
    "Pacaembu": "Zona Oeste", "Barra Funda": "Zona Oeste", "Lapa": "Zona Oeste", 
    "Alto da Lapa": "Zona Oeste", "Vila Leopoldina": "Zona Oeste", "Vila Romana": "Zona Oeste",
    "Butantã": "Zona Oeste", "Jaguaré": "Zona Oeste", "Morumbi": "Zona Oeste", 
    "Panamby": "Zona Oeste", "Real Parque": "Zona Oeste", "Vila Sônia": "Zona Oeste",
    
    # Zona Sul
    "Vila Mariana": "Zona Sul", "Paraíso": "Zona Sul", "Vila Clementino": "Zona Sul",
    "Chácara Klabin": "Zona Sul", "Saúde": "Zona Sul", "Moema": "Zona Sul", 
    "Indianópolis": "Zona Sul", "Campo Belo": "Zona Sul", "Brooklin": "Zona Sul", 
    "Brooklin Paulista": "Zona Sul", "Vila Mascote": "Zona Sul", "Santo Amaro": "Zona Sul", 
    "Chácara Santo Antônio": "Zona Sul", "Alto da Boa Vista": "Zona Sul", "Granja Julieta": "Zona Sul",
    "Jabaquara": "Zona Sul", "Cursino": "Zona Sul", "Ipiranga": "Zona Sul", 
    "Sacomã": "Zona Sul", "Campo Limpo": "Zona Sul", "Capão Redondo": "Zona Sul",
    "Socorro": "Zona Sul", "Interlagos": "Zona Sul", "Cidade Dutra": "Zona Sul",
    
    # Zona Norte
    "Santana": "Zona Norte", "Jardim São Paulo": "Zona Norte", "Alto de Santana": "Zona Norte",
    "Tucuruvi": "Zona Norte", "Parada Inglesa": "Zona Norte", "Mandaqui": "Zona Norte",
    "Tremembé": "Zona Norte", "Jaçanã": "Zona Norte", "Casa Verde": "Zona Norte", 
    "Limão": "Zona Norte", "Freguesia do Ó": "Zona Norte", "Brasilândia": "Zona Norte",
    "Pirituba": "Zona Norte", "Jaraguá": "Zona Norte", "Vila Maria": "Zona Norte",
    "Vila Guilherme": "Zona Norte", "Vila Medeiros": "Zona Norte",
    
    # Zona Leste
    "Mooca": "Zona Leste", "Alto da Mooca": "Zona Leste", "Tatuapé": "Zona Leste", 
    "Jardim Anália Franco": "Zona Leste", "Anália Franco": "Zona Leste", "Belém": "Zona Leste",
    "Água Rasa": "Zona Leste", "Vila Formosa": "Zona Leste", "Carrão": "Zona Leste", 
    "Vila Carrão": "Zona Leste", "Penha": "Zona Leste", "Vila Matilde": "Zona Leste", 
    "Artur Alvim": "Zona Leste", "Itaquera": "Zona Leste", "Cidade Líder": "Zona Leste",
    "São Miguel Paulista": "Zona Leste", "Itaim Paulista": "Zona Leste", "Guaianases": "Zona Leste",
    "São Mateus": "Zona Leste", "Sapopemba": "Zona Leste", "Vila Prudente": "Zona Leste",
    "Parque da Mooca": "Zona Leste", "Vila Zelina": "Zona Leste"
}

def classificar_regiao(row):
    dist = str(row['district']).strip()
    addr = str(row['address']).strip().lower()
    
    # Match exato no dicionário
    if dist in MAPA_REGIOES:
        return MAPA_REGIOES[dist]
        
    # Match por pistas textuais no endereço ou distrito
    comb = f"{dist.lower()} {addr}"
    if any(k in comb for k in ["zona sul", "(z sul)", "sacoma", "ipiranga", "jabaquara", "interlagos"]):
        return "Zona Sul"
    if any(k in comb for k in ["zona leste", "(z leste)", "tatuape", "mooca", "itaquera", "penha"]):
        return "Zona Leste"
    if any(k in comb for k in ["zona norte", "(z norte)", "santana", "tucuruvi", "casa verde", "freguesia"]):
        return "Zona Norte"
    if any(k in comb for k in ["zona oeste", "(z oeste)", "pinheiros", "perdizes", "lapa", "butanta"]):
        return "Zona Oeste"
    if any(k in comb for k in ["centro", "se ", "republica", "bela vista", "consolacao"]):
        return "Centro"
        
    # Municípios da Grande SP comuns no dataset
    if any(m in comb for m in ["alphaville", "barueri", "osasco", "guarulhos", "santo andre", "sao bernardo", "sao caetano", "diadema", "cotia"]):
        return "Região Metropolitana"
        
    return "Outras Regiões"

df_clean['regiao'] = df_clean.apply(classificar_regiao, axis=1)

print("Distribuição de Anúncios por Região de São Paulo:")
print(df_clean['regiao'].value_counts())


### 5. Análise Estatística Descritiva: Média vs. Mediana e Assimetria Positiva

Em dados imobiliários, a **distribuição de preços apresenta assimetria positiva à direita**: imóveis de altíssimo padrão (coberturas e mansões com aluguéis acima de R$ 20.000) puxam a **média aritmética para cima**, tornando a **mediana** o indicador de tendência central mais fidedigno para a realidade da população.


In [ ]:
# Resumo Estatístico Geral
stats_aluguel = df_clean['rent_price'].describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95])
media_val = df_clean['rent_price'].mean()
mediana_val = df_clean['rent_price'].median()
desvio_val = df_clean['rent_price'].std()

print("="*60)
print("📊 RESUMO ESTATÍSTICO DO VALOR DO ALUGUEL (SÃO PAULO)")
print("="*60)
print(f"Total de Anúncios Auditados : {len(df_clean):,}")
print(f"Média Aritmética             : R$ {media_val:,.2f}")
print(f"Mediana (50% dos imóveis)   : R$ {mediana_val:,.2f}")
print(f"Desvio Padrão                : R$ {desvio_val:,.2f}")
print(f"Diferença Média x Mediana    : R$ {media_val - mediana_val:,.2f} (+{(media_val/mediana_val - 1)*100:.1f}%)")
print(f"Valor Mínimo Registrado      : R$ {df_clean['rent_price'].min():,.2f}")
print(f"Valor Máximo Registrado      : R$ {df_clean['rent_price'].max():,.2f}")
print(f"Primeiro Quartil (Q1 - 25%)  : R$ {stats_aluguel['25%']:,.2f}")
print(f"Terceiro Quartil (Q3 - 75%)  : R$ {stats_aluguel['75%']:,.2f}")
print("="*60)


### 6. Gráfico 1: Aluguel Médio vs. Mediano por Tipo de Imóvel

In [ ]:
# Agregação por Tipo
tipo_agg = df_clean.groupby('type')['rent_price'].agg(['count', 'mean', 'median', 'std']).reset_index()
tipo_agg.columns = ['Tipo', 'Quantidade', 'Media', 'Mediana', 'Desvio_Padrao']

# Visualização Interativa com Plotly
fig1 = go.Figure()
fig1.add_trace(go.Bar(
    x=tipo_agg['Tipo'],
    y=tipo_agg['Media'],
    name='Média (R$)',
    marker_color='#2563eb',
    text=[f"R$ {v:,.0f}" for v in tipo_agg['Media']],
    textposition='outside'
))
fig1.add_trace(go.Bar(
    x=tipo_agg['Tipo'],
    y=tipo_agg['Mediana'],
    name='Mediana (R$)',
    marker_color='#10b981',
    text=[f"R$ {v:,.0f}" for v in tipo_agg['Mediana']],
    textposition='outside'
))

fig1.update_layout(
    title='<b>Gráfico 1 — Comparativo: Preço Médio vs. Mediano por Tipo de Imóvel</b><br><sup>Demonstração da assimetria positiva entre média e mediana em São Paulo (n = 11.655)</sup>',
    barmode='group',
    xaxis_title='Tipologia do Imóvel',
    yaxis_title='Valor Mensal (R$)',
    template='plotly_white',
    height=480,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig1.show()
display(tipo_agg)


### 7. Gráfico 2: Aluguel e Preço/m² por Macrozona Geográfica

In [ ]:
# Agregação por Região
regiao_agg = df_clean.groupby('regiao').agg(
    qtd=('rent_price', 'count'),
    media_aluguel=('rent_price', 'mean'),
    mediana_aluguel=('rent_price', 'median'),
    media_m2=('preco_m2', 'mean'),
    mediana_m2=('preco_m2', 'median')
).reset_index().sort_values('mediana_aluguel', ascending=False)

fig2 = px.bar(
    regiao_agg,
    x='regiao',
    y='mediana_aluguel',
    color='regiao',
    text=regiao_agg['mediana_aluguel'].apply(lambda x: f"R$ {x:,.0f}"),
    title='<b>Gráfico 2 — Aluguel Mediano por Região de São Paulo</b><br><sup>Zona Oeste lidera em valor mediano absoluto, seguida pela Zona Sul e Centro</sup>',
    labels={'regiao': 'Região / Macrozona', 'mediana_aluguel': 'Aluguel Mediano (R$)'},
    template='plotly_white',
    height=450
)
fig2.update_traces(textposition='outside')
fig2.show()
display(regiao_agg)


### 8. Gráfico 3: Impacto de Quartos e Vagas de Garagem na Precificação

In [ ]:
# Filtro de imóveis até 5 quartos e 4 vagas para garantir significância estatística
df_sub = df_clean[(df_clean['bedrooms'] <= 5) & (df_clean['garage'] <= 4)]

fig3 = px.box(
    df_sub,
    x='bedrooms',
    y='rent_price',
    color='garage',
    title='<b>Gráfico 3 — Distribuição do Aluguel por Quartos e Vagas de Garagem</b><br><sup>Vagas de garagem exercem forte efeito multiplicador no valor final da locação</sup>',
    labels={'bedrooms': 'Número de Quartos', 'rent_price': 'Valor do Aluguel (R$)', 'garage': 'Vagas'},
    template='plotly_white',
    height=500
)
fig3.update_yaxes(range=[0, 15000]) # Foco visual para melhor leitura dos quartis
fig3.show()


### 9. Gráfico 4: Dispersão e Correlação entre Área Útil ($m^2$) e Valor do Aluguel

In [ ]:
# Gráfico de dispersão com amostra ou visualização interativa
fig4 = px.scatter(
    df_clean[df_clean['area'] <= 400],
    x='area',
    y='rent_price',
    color='type',
    trendline='ols',
    opacity=0.6,
    hover_data=['district', 'bedrooms', 'garage'],
    title='<b>Gráfico 4 — Relação entre Área Útil (m²) e Valor do Aluguel</b><br><sup>Tendência linear positiva com alta dispersão gerada pelo fator localização</sup>',
    labels={'area': 'Área Útil (m²)', 'rent_price': 'Aluguel Mensal (R$)', 'type': 'Tipo'},
    template='plotly_white',
    height=520
)
fig4.update_yaxes(range=[0, 25000])
fig4.show()

# Correlação de Pearson
corr_area = df_clean['area'].corr(df_clean['rent_price'])
corr_vagas = df_clean['garage'].corr(df_clean['rent_price'])
corr_quartos = df_clean['bedrooms'].corr(df_clean['rent_price'])
print(f"Correlação de Pearson com o Preço do Aluguel:")
print(f"• Área Útil : {corr_area:.3f} (Forte relação)")
print(f"• Vagas     : {corr_vagas:.3f} (Forte relação)")
print(f"• Quartos   : {corr_quartos:.3f} (Moderada relação)")


### 10. Gráfico 5: Rankings dos Bairros Mais Caros e Mais Baratos de São Paulo

> ⚠️ **Auditoria de Representatividade Estatística:**
> No dataset original, 340 bairros possuem apenas 1 anúncio cadastrado. Para evitar distorções estatísticas decorrentes de amostras minúsculas, aplicou-se o filtro rigoroso de **mínimo de 15 anúncios ($n \ge 15$)** para qualificação no ranking oficial.


In [ ]:
# Agrupamento por bairro
bairros_stats = df_clean.groupby(['district', 'regiao']).agg(
    n_anuncios=('rent_price', 'count'),
    aluguel_medio=('rent_price', 'mean'),
    aluguel_mediano=('rent_price', 'median'),
    preco_m2_mediano=('preco_m2', 'median')
).reset_index()

# Filtro de representatividade (n >= 15)
bairros_qualificados = bairros_stats[bairros_stats['n_anuncios'] >= 15].copy()

top10_caros = bairros_qualificados.sort_values('aluguel_mediano', ascending=False).head(10)
top10_baratos = bairros_qualificados.sort_values('aluguel_mediano', ascending=True).head(10)

print(f"Total de Bairros no Dataset : {len(bairros_stats)}")
print(f"Bairros Qualificados (n>=15): {len(bairros_qualificados)}")

# Visualização Top 10 Mais Caros
fig5a = px.bar(
    top10_caros,
    x='aluguel_mediano',
    y='district',
    color='regiao',
    orientation='h',
    text=top10_caros['aluguel_mediano'].apply(lambda x: f"R$ {x:,.0f}"),
    title='<b>Top 10 Bairros Mais Caros de São Paulo (Mediana, n >= 15)</b><br><sup>Predomínio absoluto da Zona Oeste e eixos corporativos</sup>',
    labels={'district': 'Bairro / Distrito', 'aluguel_mediano': 'Aluguel Mediano (R$)'},
    template='plotly_white',
    height=420
)
fig5a.update_layout(yaxis={'categoryorder':'total ascending'})
fig5a.show()

# Visualização Top 10 Mais Acessíveis
fig5b = px.bar(
    top10_baratos,
    x='aluguel_mediano',
    y='district',
    color='regiao',
    orientation='h',
    text=top10_baratos['aluguel_mediano'].apply(lambda x: f"R$ {x:,.0f}"),
    title='<b>Top 10 Bairros Mais Acessíveis de São Paulo (Mediana, n >= 15)</b><br><sup>Predomínio da Zona Leste e extremos da Zona Norte</sup>',
    labels={'district': 'Bairro / Distrito', 'aluguel_mediano': 'Aluguel Mediano (R$)'},
    template='plotly_white',
    height=420
)
fig5b.update_layout(yaxis={'categoryorder':'total descending'})
fig5b.show()


### 11. Visualização Cartográfica e Mapa Interativo (Folium)

Mapa georreferenciado interativo da cidade de São Paulo, plotando os principais bairros com círculos dimensionados e coloridos de acordo com o valor mediano do aluguel.


In [ ]:
# Coordenadas aproximadas dos principais bairros representativos
coords_dict = {
    "Itaim Bibi": [-23.5839, -46.6789],
    "Pinheiros": [-23.5617, -46.7028],
    "Jardim Paulista": [-23.5694, -46.6631],
    "Vila Olímpia": [-23.5956, -46.6869],
    "Moema": [-23.6042, -46.6617],
    "Perdizes": [-23.5369, -46.6744],
    "Bela Vista": [-23.5606, -46.6493],
    "Consolação": [-23.5518, -46.6582],
    "Vila Mariana": [-23.5896, -46.6347],
    "Santana": [-23.5011, -46.6253],
    "Tatuapé": [-23.5406, -46.5767],
    "Mooca": [-23.5558, -46.5989],
    "Ipiranga": [-23.5919, -46.6033],
    "Saúde": [-23.6156, -46.6397],
    "Morumbi": [-23.6067, -46.7214],
    "Butantã": [-23.5719, -46.7081],
    "Itaquera": [-23.5381, -46.4533],
    "São Mateus": [-23.6025, -46.4789],
    "Brasilândia": [-23.4689, -46.6853],
    "Pirituba": [-23.4889, -46.7289]
}

# Criação do mapa Folium centrado em São Paulo
mapa_sp = folium.Map(
    location=[-23.5505, -46.6333],
    zoom_start=11,
    tiles="CartoDB positron"
)

def get_cor_aluguel(valor):
    if valor >= 5000:
        return '#dc2626' # Vermelho (Altíssimo)
    elif valor >= 3000:
        return '#f97316' # Laranja (Médio-Alto)
    elif valor >= 2000:
        return '#3b82f6' # Azul (Médio)
    else:
        return '#10b981' # Verde (Econômico)

# Adiciona círculos de marcadores
for _, row in bairros_qualificados.iterrows():
    nome = row['district']
    if nome in coords_dict:
        lat, lon = coords_dict[nome]
        val = row['aluguel_mediano']
        raio = max(6, min(24, val / 250))
        
        popup_html = f"""
        <div style='font-family: Arial; font-size: 13px; min-width: 170px;'>
            <h4 style='margin:0 0 5px 0; color:#1e293b;'><b>{nome}</b></h4>
            <b>Região:</b> {row['regiao']}<br>
            <b>Aluguel Mediano:</b> R$ {val:,.2f}<br>
            <b>Preço/m² Mediano:</b> R$ {row['preco_m2_mediano']:,.2f}<br>
            <b>Anúncios:</b> {row['n_anuncios']}
        </div>
        """
        
        folium.CircleMarker(
            location=[lat, lon],
            radius=raio,
            popup=folium.Popup(popup_html, max_width=300),
            tooltip=f"{nome} — Mediana: R$ {val:,.0f}",
            color=get_cor_aluguel(val),
            fill=True,
            fill_color=get_cor_aluguel(val),
            fill_opacity=0.75,
            weight=1.5
        ).add_to(mapa_sp)

mapa_sp


### 12. Resposta à Pergunta Orientadora do Edital SYNAPTA

> *"Quais características parecem acompanhar os maiores e os menores preços de aluguel e o que os dados não permitem concluir?"*

---

#### 🟢 1. O que acompanha os MAIORES preços de aluguel:
1. **Localização Geográfica Privilegiada (Eixo Sudoeste/Zona Oeste):**
   - Bairros como **Itaim Bibi, Pinheiros, Vila Nova Conceição e Jardins** concentram as maiores medianas (superiores a R$ 6.000 mensais).
   - A Macrozona Oeste apresenta valor mediano quase **2,5x superior** ao da Zona Leste.
2. **Vagas de Garagem (Efeito Multiplicador):**
   - A correlação entre vagas e preço ($r = 0,55$) é até mais impactante do que o número de quartos isolado, pois reflete tipologias de alto padrão e condomínios com infraestrutura nobre.
3. **Área Útil e R$/m²:**
   - Imóveis acima de $150 m^2$ concentram-se quase que exclusivamente nos quartis superiores de preço.

---

#### 🔵 2. O que acompanha os MENORES preços de aluguel:
1. **Periferias e Extremos das Zonas Leste e Norte:**
   - Distritos como **São Mateus, Cidade Líder, Guaianases, Jaraguá e Brasilândia** apresentam as menores medianas da cidade (entre R$ 1.100 e R$ 1.500).
2. **Ausência de Garagem:**
   - Imóveis com zero vagas têm mediana de aluguel expressivamente menor.
3. **Casas Simples em Bairros Tradicionais:**
   - Casas em vilas residenciais de periferia apresentam os menores custos absolutos por metro quadrado ($R\$/m^2 \le 20$).

---

#### ⚠️ 3. O que os dados NÃO permitem concluir (Limitações Metodológicas):
1. **Preço Anunciado vs. Preço Efetivamente Transacionado:**
   - O dataset registra o valor de anúncio (*asking price*), que tipicamente embute margem de negociação de 5% a 15% em relação ao contrato fechado.
2. **Ausência de Encargos Acessórios (Condomínio e IPTU):**
   - Em São Paulo, o condomínio frequentemente representa de 20% a 40% do custo mensal total de um apartamento. O dataset não possui essas colunas.
3. **Estado de Conservação e Mobília:**
   - Apartamentos mobiliados ou reformados cobram ágios expressivos sobre imóveis no mesmo prédio; a base não informa se o imóvel é mobiliado ou novo.
4. **Viés Amostral das Plataformas de Anúncio:**
   - A base é derivada de anúncios online, o que sub-representa a locação informal nas periferias e sobre-representa bairros de classe média e alta.


### 13. (Bônus) Como Executar o Dashboard Web em Flask Diretamente no Google Colab

Caso deseje abrir a interface web completa desenvolvida para este projeto (com Mapa Leaflet e Dashboard Chart.js) diretamente de dentro do Google Colab:


In [ ]:
# Instruções para rodar o Flask diretamente no Google Colab via túnel público
"""
Para rodar a aplicação web no Colab, execute em uma nova célula:

!git clone https://github.com/gui-santos-dev/synapta.git
%cd synapta
!pip install -r requirements.txt
!pip install pyngrok

from pyngrok import ngrok
# ngrok.set_auth_token("SEU_TOKEN_NGROK")
# public_url = ngrok.connect(5000)
# print("Acesse a aplicação Web em:", public_url)
# !python app.py
"""
print("💡 Para executar o Web App no Colab, consulte as instruções comentadas acima.")
